# 0. 환경설정

## 0-1. 필요 라이브러리 설치

## 0-1-1. 설치 명령어

In [ ]:
import re
import torch
v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
!pip install --no-deps bitsandbytes==0.45.5 accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
!pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install --no-deps unsloth
!pip install transformers==4.55.4

### 0-1-2. 각 라이브러리에 대한 설명

1. **unsloth**
    - 양자화된 모델을 로드하고, 특히 LoRA 학습과정을 최적화하는 라이브러리
    - PyTorch의 기본 연산 방법을 대체하여서, `완전히 최적화된 CUDA 커널`을 사용할 수 있도록 함.
    - 이를 통해 VRAM 사용량을 줄이고, 학습 속도를 2배 이상 향상 시켜 줌.
2. **bitsandbytes**
    - QLoRA의 핵심
    - 모델의 가중치를 정해진 비트 크기로 로드하고, 학습 시에는 양자화된 가중치를 다시 16비트나 32비트와 같이 `역양자화`하는 역할을 수행
3. **`peft` (Parameter-Efficient Fine-Tuning)**
    - Hugging Face의 라이브러리
    - 원본 모델의 가중치는 **동결**시킨 채, 학습시킬 **LoRA 어댑터**를 모델의 `target_moduels`(예: QKV 레이어)에 주입하는 역할
4. **`trl` (Transformer Reinforcement Learning)**
    - `SFTTrainer` (Supervised Fine-tuning Trainer)를 제공
    - 학습에 필요한 데이터를 학습에 적절한 문자열로 변환하고 토크나이징하는 과정을 추상화 해주는 라이브러리
    - 학습 과정에 필요한 각종 **하이퍼파라미터**를 한 번에 관리 할 수 있음
        - 학습률, 학습 스케쥴러, 옵티마이저, 배치사이즈, 에포크 등
5. **transformers**
    - Hugging Face의 데이터셋 라이브러리
    - 데이터를 로드하고, 전처리를 병렬로 처리할 수 있는 라이브러리

## 0-2. 설치 과정

### 0-2-1. Colab 환경에서 성능 최적화를 목표로

- 최우선 목표는 `최대 속도 확보`에 있음.
- colab은 로컬 환경(각자 마다 서로 다른 PC 환경)과 달리 **표준화된 환경**이 제공됨
- 예를 들어, 아래와 같이 서로 완전히 상이할 수 있는 각종 환경이 Colab은 모두가 동일함.
    - GPU는 A100 혹은 T4와 같이 동일한 GPU
    - CUDA 버젼은 12.x
    - PyTorch는 2.x
- Unsloth는 **`표준화 된 Colab 환경`**은 어떤 사용자든 모두 완전히 동일한 환경이므로, 이 환경에 `미리 컴파일된` 최정과 라이브러리를 제공 할 수 있음.
1. 설치 전략
    - `--no-deps` (의존성 무시)
        - 일반적인 상황이라면, 절대 사용하지 않을 옵션
        - pip 명령어로 관련 라이브러리를 설치할 때, 특정 라이브러리가 구 버젼임으로 인해 호환되지 않는 다른 의존성 라이브러리를 설치하는 상황을 막을 수 있음
    - Unsloth가 Colab 환경에 맞춰 미리 컴파일 해 둔 `xformers`, `triton` 등의 최적화 커널을 설치
    - 이후, Unsloth와 transformers 등을 `--no-deps`로 설치하여, 이미 설치된 최적화 라이브러리와 충돌하지 않도록 할 수 있음.
2. 주의 사항
    - unsloth 기본적으로 로컬에 PyTorch가 CUDA를 사용 가능하도록 미리 환경 설정이 되어 있음을 가정하므로, 이 작업이 미리 되어 있어야 함.
    - Colab은 해당 설정이 이미 되어 있으므로, unsloth 설치만 진행하여도 문제 없음.
        - `단, 반드시 현재 PyTorch의 버전과 CUDA 버전과 일치하는 라이브러리를 설치해야 하므로 현재 버젼 확인 필수`
        - 이를 위하여, 아래의 코드가 작성되어 있는것
        - `v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)`

## 0-3. xformers와 triton이란?

### 0-3-1. xformers

- Meta에서 개발한 라이브러리
- 트랜스포머의 핵심인 어텐션 연산을 최적화하는데 특화되어 있음.
1. 핵심 기술
    - 기존 어텐션은 Q x K^T 연산을 수행할 때, (시퀀스 길이 NxN) 크기의 거대한 **어텐션 행렬**을 VRAM에 통째로 생성해야 했음.
        - 즉, 시퀀스 길이가 2048만 되어도 2048 x 2048 행렬이 필요했음.
    - `Flash Attention`은 이런 전체 계산을 수행하는 것이 아닌, 여러 개의 작은 블록 또는 타일로 나눠서 연산을 진행하도록 함.
    - 각 블록을 GPU의 고속 캐시 메모리(SRAM)에서 계산
    - 중간 결과만 VRAM에 저장 → VRAM 사용량 감소, 연산 속도 향상을 하게 해 줌
2. 주의 사항
    - 이 `xformers`는 GPU의 연산 방식에 연관하기 때문에 GPU 하드웨어애 매우 민감함
    - GPU 하드웨어에 민감 → CUDA 버젼에 민감 → PyTorch 버젼에도 `극도로 민감함`
    - 따라서, 로컬 환경 (각자 PC마다 사용하는 GPU의 버젼, 이때 사용하는 각종 라이브러리 등)이 서로 너무나도 다르기 때문에 이에 적절한 `xformers`를 찾기가 어려움.
    - 그러므로 colab을 쓰는 쪽이 속 편하다….

### 0-3-2. triton

- OpenAI에서 개발한 언어로, **파이썬과 유사한 문법으로 고성능 GPU 커널을 작성 할 수 있게 해주는 컴파일러**
- **Unsloth**가 PyTorch의 기본 연산 대신, LoRA 연산이나 4비트 역양자화와 같은 복잡한 연산과정에 특화된 연산자를 이 `Triton`으로 만들어서 사용함.
- 이 과정에서, 데이터 간의 이동을 최소화 하고, GPU를 완전히 활용하도록 설계되어 있음.
****